<hr>

# Set up


<hr>

In [1]:
# Manually adding the parent directory to Root of the project
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))



#Importing python functions for integration to snowflake and other utility functions
from py_utils import snowflake_sql, nb_html_export, nb_html_slide_export,  utils
# OR import each function inside each module in py_utils package
from py_utils import convert_notebook_to_html_string, write_notebook_to_html, convert_notebook_to_slides_html, write_notebook_to_html_slide, create_snowflake_sql_engine, load_data_try_parquet_first, calculate_standardised_rates, calculate_axis_lim, get_fiscal_year

# load dataframe packages
import pandas as pd
import polars as pl

# load statistical automation package
#import ph_inequalities_statistical_comparison as phsc

# set capacity to see entire dataframe

# show more (or all) columns
pl.Config.set_tbl_cols(-1)      # -1 = no column limit [web:10][web:12]

# show more (or all) rows
pl.Config.set_tbl_rows(-1)      # -1 = no row limit [web:7][web:12]

# optional: avoid truncating long strings
pl.Config.set_fmt_str_lengths(200)  # or a larger value [web:1][web:12]



polars.config.Config

<hr>

# Query


<hr>

In [2]:
# Make sure to setup your connections.toml file before running this

# This creates the Snowflake engine that will be used to run any query on snowflake
SNOWFLAKE_ENGINE = create_snowflake_sql_engine('prd')

# This is how you load the data once SNOWFLAKE ENGINE is up.
df = load_data_try_parquet_first(SNOWFLAKE_ENGINE, parquet_file_name='Data/data.parquet', sql_path='../SQL/query.sql')

df = pl.from_pandas(df)

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/f789227e-d70c-4029-b108-98ba67202f84/saml2?SAMLRequest=lZJRb9sgGEX%2FisWebYOXJTFKUqX1qnlLW6tJO6lvxOCEFYPLh%2BP034%2FYjdQ9tNLeLHwuHLjf7OJYq%2BAgLEij54hEGAVCl4ZLvZujh811OEUBOKY5U0aLOXoVgC4WM2C1auiydXt9L15aAS7wG2mg%2FY85aq2mhoEEqlktgLqSrpc3K5pEmDbWOFMahd5FPk8wAGGdNzxHOEivt3euoXHcdV3UfY2M3cUJxjjGaeypE%2FLlzB%2F9nT7gSYxHJ94THi%2Fe3C6lHp7gM63tAAH9sdkUYXG33qBgeVa9MhraWti1sAdZiof71SAA3kCWwJlj%2B3YbNpZHoE1XKfYsSlM3rfNbRv4rrgSPldlJf%2Bs8m6PmWfKJ%2Bv1rqrZ38HSAUeGEKf5kj7cVP2YvHSbHrBlX33PQq8ufeYmCx3OtyanWHKAVuT6V6fwSTsYhwSFJNoRQ8o2SUZTi9AkFmS9Taub65Nm494hqWVoDpnJGK6nFYDmZpkkyESGf4DIc4SQNtwRPw3S6ZeOJP6aajuJTxQkaxob2InbxX48xi99H38bv1jeSZ4VRsnwNro2tmfu4MBKRfkXysOpRKmom1ZJzKwB8cUqZ7soK5vyUO9sKFC%2BGU%2F%2Bd88Vf&RelayState=ver%3A3-hint%3A468572346519558-ETMsDgAAAaEv0u1dABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEKWiafJq5LSNQZvKdkFO6xsAAACgOzHcWXDamDU3Nx%2BE6uXM0ujeF8z%2B

quick overview of data

In [3]:
df.describe()

statistic,PatientPseudonym,OnWaitList,Is18Weeks,Is52Weeks,Age,Gender,EthnicityCode,GeneralPracticeCode,GeneralPracticePlaceCode,GeneralPracticePlaceName,LocationLSOACode2021,LocationIMDDecile,Core20CategoryList,C20indicator
str,str,f64,f64,f64,f64,str,str,str,str,str,str,f64,str,str
"""count""","""169808""",169808.0,169808.0,169808.0,169808.0,"""169808""","""169808""","""169808""","""169808""","""169808""","""169808""",169269.0,"""169808""","""169808"""
"""null_count""","""0""",0.0,0.0,0.0,0.0,"""0""","""0""","""0""","""0""","""0""","""0""",539.0,"""0""","""0"""
"""mean""",null,1.0,0.411918,0.023008,53.856391,null,null,null,null,null,null,5.466654,null,null
"""std""",null,0.0,0.492182,0.14993,23.300817,null,null,null,null,null,null,2.383945,null,null
"""min""","""00001FE581FDF65B28C98F26F5A3E29F6DAC487D84E11F4FC1872C4FEC68D3DC""",1.0,0.0,0.0,0.0,"""Female""","""*""","""D82001""","""06L""","""Great Yarmouth and Waveney""","""*""",1.0,"""ASTHMA""","""C20"""
"""25%""",null,1.0,0.0,0.0,37.0,null,null,null,null,null,null,4.0,null,null
"""50%""",null,1.0,0.0,0.0,58.0,null,null,null,null,null,null,5.0,null,null
"""75%""",null,1.0,1.0,0.0,73.0,null,null,null,null,null,null,7.0,null,null
"""max""","""FFFFFBC30EE6B57DD6D5B3D691083B549DB68C4A2DB53BDC4DC4DC7175978A78""",1.0,1.0,1.0,108.0,"""Not Recorded""","""Z""","""Y06275""","""07K""","""West Suffolk""","""W01002002""",10.0,"""VET""","""Not C20"""


<hr>

# Proof-Of-Concept: patients waiting 18 weeks or less


<hr>

In [4]:
import ph_inequalities_statistical_comparison as phsc

##  C20 vs not: overall

In [5]:
automated_output = phsc.crude_proportion_df(
    df,
    event_col= "Is18Weeks",
    inequalities_cols= ["C20indicator"]
)

automated_output

C20indicator,events,n,proportion,lower,upper,confidence,method,notes,significance
str,f64,f64,f64,f64,f64,f64,str,str,str
"""C20""",9431.0,22032.0,0.428059,0.421539,0.434605,0.95,"""Wilson score""","""""","""Higher"""
"""Not C20""",60516.0,147776.0,0.409512,0.407007,0.412021,0.95,"""Wilson score""","""""","""Not significant"""
"""All""",69947.0,169808.0,0.411918,0.409579,0.414261,0.95,"""Wilson score""","""""","""Reference"""


# C20 crossed with Gender: inequalities lens' include overalls

In [6]:
automated_output = phsc.crude_proportion_df(
    df,
    event_col= "Is18Weeks",
    inequalities_cols= ["C20indicator", "Gender"]
)

# suppressing information for public repo

(
    automated_output
    .with_columns(
        pl.when(pl.col("n") < 5).then(None).otherwise(pl.col("n")).alias("n"),
        pl.when(pl.col("events") < 5).then(None).otherwise(pl.col("events")).alias("events")
    )
).head(100)

C20indicator,Gender,events,n,proportion,lower,upper,confidence,method,notes,significance
str,str,f64,f64,f64,f64,f64,f64,str,str,str
"""C20""","""Female""",5472.0,12688.0,0.431274,0.422678,0.439911,0.95,"""Wilson score""","""""","""Higher"""
"""C20""","""Male""",3958.0,9341.0,0.423723,0.413736,0.433774,0.95,"""Wilson score""","""""","""Higher"""
"""C20""","""Not Recorded""",null,null,0.333333,0.061492,0.79234,0.95,"""Wilson score""","""Low event count (<10) - must flag as unstable or suppress in visual outputs. | Low non-event count (<10) - must flag as unstable or suppress in visual outputs. | Low sample size (<40) - flag the propo…","""Not significant"""
"""Not C20""","""Female""",34342.0,83416.0,0.411696,0.40836,0.415039,0.95,"""Wilson score""","""""","""Not significant"""
"""Not C20""","""Male""",26165.0,64338.0,0.40668,0.40289,0.410481,0.95,"""Wilson score""","""""","""Lower"""
"""Not C20""","""Not Recorded""",9.0,22.0,0.409091,0.232558,0.612652,0.95,"""Wilson score""","""Low event count (<10) - must flag as unstable or suppress in visual outputs. | Low sample size (<40) - flag the proportion as unstable.""","""Not significant"""
"""C20""","""All""",9431.0,22032.0,0.428059,0.421539,0.434605,0.95,"""Wilson score""","""""","""Higher"""
"""Not C20""","""All""",60516.0,147776.0,0.409512,0.407007,0.412021,0.95,"""Wilson score""","""""","""Not significant"""
"""All""","""Female""",39814.0,96104.0,0.41428,0.411169,0.417398,0.95,"""Wilson score""","""""","""Not significant"""


##  C20 vs not: organisational heirarchy for place and general practice

The code below demonstrates how, for organisations, a hierarchy can be defined (where the top level categories do not share elements of lower categories).

Critically, this is implemented so that more than one level of the heirarchy can be specified without creating a degenerate level (e.g a practice in Ipswich being crossed with Central Norfolk place)



In [7]:
automated_output = phsc.crude_proportion_df(
    df,
    event_col = "Is18Weeks",
    inequalities_cols= ["C20indicator"],
    organisational_cols={
        "org": ["GeneralPracticePlaceName", "GeneralPracticeCode"]
    }
)

# suppressing information for public repo: general practice code, event count, denominator

(
    automated_output
    .with_columns(
        pl.col("GeneralPracticeCode").hash(seed=42),
        pl.when(pl.col("n") < 5).then(None).otherwise(pl.col("n")).alias("n"),
        pl.when(pl.col("events") < 5).then(None).otherwise(pl.col("events")).alias("events")
    )
).head(100)



GeneralPracticePlaceName,GeneralPracticeCode,C20indicator,events,n,proportion,lower,upper,confidence,method,notes,significance
str,u64,str,f64,f64,f64,f64,f64,f64,str,str,str
"""Great Yarmouth and Waveney""",15398788212864414384,"""C20""",319.0,708.0,0.450565,0.41428,0.487383,0.95,"""Wilson score""","""""","""Higher"""
"""Great Yarmouth and Waveney""",15398788212864414384,"""Not C20""",803.0,1895.0,0.423747,0.401674,0.446128,0.95,"""Wilson score""","""""","""Not significant"""
"""Great Yarmouth and Waveney""",9302709536925711100,"""C20""",729.0,1681.0,0.43367,0.410157,0.457486,0.95,"""Wilson score""","""""","""Not significant"""
"""Great Yarmouth and Waveney""",9302709536925711100,"""Not C20""",674.0,1534.0,0.439374,0.41472,0.464331,0.95,"""Wilson score""","""""","""Higher"""
"""Great Yarmouth and Waveney""",2429350749438711605,"""C20""",130.0,281.0,0.462633,0.405232,0.521042,0.95,"""Wilson score""","""""","""Not significant"""
"""Great Yarmouth and Waveney""",2429350749438711605,"""Not C20""",906.0,2015.0,0.449628,0.428024,0.471423,0.95,"""Wilson score""","""""","""Higher"""
"""Great Yarmouth and Waveney""",10933599869271199946,"""C20""",8.0,20.0,0.4,0.218807,0.613418,0.95,"""Wilson score""","""Low event count (<10) - must flag as unstable or suppress in visual outputs. | Low sample size (<40) - flag the proportion as unstable.""","""Not significant"""
"""Great Yarmouth and Waveney""",10933599869271199946,"""Not C20""",996.0,2303.0,0.432479,0.412375,0.452809,0.95,"""Wilson score""","""""","""Higher"""
"""Great Yarmouth and Waveney""",4476549231349469476,"""C20""",556.0,1221.0,0.455364,0.427615,0.483394,0.95,"""Wilson score""","""""","""Higher"""


## C20 vs not: age standardized across general practice

Again, this would be an innapropriate use, but for testing this will reveal the python outputs.

In [8]:
automated_output = phsc.directly_standardized_proportion_df(
    df = df,
    event_col= "Is18Weeks",
    strata_cols= ["Age"],
    inequalities_cols= ["C20indicator"],
    organisational_cols={
        "org": ["GeneralPracticePlaceName", "GeneralPracticeCode"]
    }
)

# suppressing information for public repo: general practice code, event count, denominator

(
    automated_output
    .with_columns(
        pl.col("GeneralPracticeCode").hash(seed=42),
        pl.when(pl.col("n") < 5).then(None).otherwise(pl.col("n")).alias("n"),
        pl.when(pl.col("events") < 5).then(None).otherwise(pl.col("events")).alias("events")
    )
).head(100)

GeneralPracticePlaceName,GeneralPracticeCode,C20indicator,events,n,dsp,dsp_lower,dsp_upper,confidence,method,notes,significance
str,u64,str,i64,i64,f64,f64,f64,f64,str,str,str
"""Great Yarmouth and Waveney""",15398788212864414384,"""C20""",319,708,0.446271,0.409283,0.484719,0.95,"""Wilson-MOVER""","""""","""Not significant"""
"""Great Yarmouth and Waveney""",15398788212864414384,"""Not C20""",803,1895,0.423883,0.40207,0.446307,0.95,"""Wilson-MOVER""","""""","""Not significant"""
"""Great Yarmouth and Waveney""",9302709536925711100,"""C20""",729,1681,0.427847,0.401398,0.455434,0.95,"""Wilson-MOVER""","""""","""Not significant"""
"""Great Yarmouth and Waveney""",9302709536925711100,"""Not C20""",674,1534,0.439825,0.4154,0.464843,0.95,"""Wilson-MOVER""","""""","""Higher"""
"""Great Yarmouth and Waveney""",2429350749438711605,"""C20""",130,281,0.451104,0.393153,0.512987,0.95,"""Wilson-MOVER""","""""","""Not significant"""
"""Great Yarmouth and Waveney""",2429350749438711605,"""Not C20""",906,2015,0.45054,0.429118,0.472326,0.95,"""Wilson-MOVER""","""""","""Higher"""
"""Great Yarmouth and Waveney""",10933599869271199946,"""C20""",8,20,0.383026,0.247422,0.573713,0.95,"""Wilson-MOVER""","""Unreliable: at least one stratum has n < 10 flag as unstable or suppress in visual outputs and strongly consider coarsening strata, inequalities or dimensions | Unreliable: stratum non-event count < 1…","""Not significant"""
"""Great Yarmouth and Waveney""",10933599869271199946,"""Not C20""",996,2303,0.435075,0.4147,0.45588,0.95,"""Wilson-MOVER""","""""","""Higher"""
"""Great Yarmouth and Waveney""",4476549231349469476,"""C20""",556,1221,0.441582,0.412697,0.471783,0.95,"""Wilson-MOVER""","""""","""Higher"""


You will notice for certain rows, the confidence interval and dsp exactly match. This is because there is a 100% proportion